# VideoToSMPL — Notebook 01: Video → SMPL (GVHMR)

**What this notebook does:** Takes an RGB video and returns SMPL parameters (`hmr4d_results.pt`) using [GVHMR](https://github.com/zju3dv/GVHMR).

**Runtime:** ~2 minutes/video on T4. Self-contained — no private repo access needed.

1. **Runtime** → **Change runtime type** → **T4 GPU**. Then **Run all**.
2. Upload your video when prompted.
3. Download the resulting `.pt` + SMPL JSON at the end.

> Something break? See the [troubleshooting guide](https://bhaveshbakshi633.github.io/VideoToSMPL/docs/troubleshooting/).

## 1. GPU sanity check

In [ ]:
import subprocess, sys, shutil

assert shutil.which('nvidia-smi'), 'No GPU detected. Runtime → Change runtime type → T4 GPU.'
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader']).decode())
import torch
assert torch.cuda.is_available(), 'torch does not see CUDA. Restart runtime.'
print(f'Python {sys.version.split()[0]} · torch {torch.__version__} · CUDA {torch.version.cuda}')

## 2. Clone GVHMR + install deps

GVHMR is the only external repo needed. All weights live on HuggingFace.

In [ ]:
%cd /content
![ -d GVHMR ] || git clone --depth 1 https://github.com/zju3dv/GVHMR.git
%cd /content/GVHMR
!pip install -q --upgrade pip
!pip install -q -r requirements.txt
!pip install -q -e .
print('GVHMR installed.')

## 3. Download model weights

GVHMR + ViTPose + HMR2 + YOLO + SMPL (~2 GB total).

In [ ]:
import os, urllib.request
from pathlib import Path

CKPT = Path('/content/GVHMR/inputs/checkpoints')
for sub in ['gvhmr', 'hmr2', 'vitpose', 'yolo', 'body_models/smpl', 'body_models/smplx']:
    (CKPT / sub).mkdir(parents=True, exist_ok=True)

WEIGHTS = {
    'gvhmr/gvhmr_siga24_release.ckpt': 'https://huggingface.co/camenduru/GVHMR/resolve/main/gvhmr/gvhmr_siga24_release.ckpt',
    'hmr2/epoch=10-step=25000.ckpt':   'https://huggingface.co/camenduru/GVHMR/resolve/main/hmr2/epoch%3D10-step%3D25000.ckpt',
    'vitpose/vitpose-h-multi-coco.pth': 'https://huggingface.co/camenduru/GVHMR/resolve/main/vitpose/vitpose-h-multi-coco.pth',
    'yolo/yolov8x.pt':                  'https://huggingface.co/camenduru/GVHMR/resolve/main/yolo/yolov8x.pt',
    'body_models/smpl/SMPL_NEUTRAL.pkl':'https://huggingface.co/camenduru/SMPLer-X/resolve/main/SMPL_NEUTRAL.pkl',
}

def download(dest: Path, url: str):
    if dest.exists() and dest.stat().st_size > 1024:
        print(f'  ✓ cached {dest.name}')
        return
    print(f'  ↓ {dest.name}', end=' ', flush=True)
    urllib.request.urlretrieve(url, dest)
    print(f'({dest.stat().st_size/1e6:.0f} MB)')

for rel, url in WEIGHTS.items():
    download(CKPT / rel, url)
print('All weights ready.')

## 4. Upload your video

Single person, full body visible. ≤1 minute works best on the free tier.

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
assert uploaded, 'No file uploaded.'
name = list(uploaded.keys())[0]
video_path = Path('/content') / name
Path('/content/GVHMR/inputs').mkdir(exist_ok=True)
video_path.write_bytes(uploaded[name])
size_mb = video_path.stat().st_size / 1e6
print(f'Uploaded: {video_path.name}  ({size_mb:.1f} MB)')

## 5. Run GVHMR extraction

In [ ]:
%cd /content/GVHMR
import subprocess, time
t0 = time.time()
result = subprocess.run(
    ['python', 'tools/demo/demo.py', '--video', str(video_path), '-s'],
    capture_output=True, text=True
)
print('stdout (last 20 lines):')
print('\n'.join(result.stdout.splitlines()[-20:]))
if result.returncode != 0:
    print('\nstderr:\n' + result.stderr)
    raise RuntimeError(f'GVHMR failed with code {result.returncode}')

pt_path = Path('/content/GVHMR/outputs/demo') / video_path.stem / 'hmr4d_results.pt'
assert pt_path.exists(), f'Result not found: {pt_path}'
print(f'\n✓ Extraction done in {time.time()-t0:.0f}s → {pt_path}')

## 6. Export SMPL params as JSON (portable)

In [ ]:
import json, torch
from pathlib import Path

blob = torch.load(pt_path, map_location='cpu', weights_only=False)
g = blob['smpl_params_global']
summary = {
    'source_video': video_path.name,
    'frames': int(g['body_pose'].shape[0]),
    'fps_estimate': 30,
    'body_pose_shape': list(g['body_pose'].shape),
    'betas_shape':     list(g['betas'].shape),
    'keys':            sorted(blob.keys()),
}
out_json = Path('/content/smpl_params.json')
out_json.write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

## 7. Download results

In [ ]:
from google.colab import files
files.download(str(pt_path))
files.download(str(out_json))
print('Downloads started.')

## Next steps

- **Retarget to Unitree G1:** run [Notebook 02](https://colab.research.google.com/github/bhaveshbakshi633/VideoToSMPL/blob/main/notebooks/02_smpl_to_g1.ipynb) with your `.pt`.
- **End-to-end:** [Notebook 03](https://colab.research.google.com/github/bhaveshbakshi633/VideoToSMPL/blob/main/notebooks/03_full_pipeline.ipynb) runs everything in one pass.
- **Run locally:** [local setup guide](https://bhaveshbakshi633.github.io/VideoToSMPL/docs/local-setup/).